In [ ]:
!pip install -q transformers accelerate datasets scikit-learn openpyxl

In [ ]:
import os
import torch
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

from transformers import (
    MambaModel,
    MambaPreTrainedModel
)

import torch.nn as nn

print("All libraries imported successfully!")

All libraries imported successfully!


In [ ]:
print("="*60)

print("PyTorch:", torch.__version__)

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("="*60)

PyTorch: 2.11.0+cu128
CUDA Available: True
GPU: Tesla T4


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving sinhala_youtube_comments_for_annotation.xlsx to sinhala_youtube_comments_for_annotation.xlsx


In [ ]:
df = pd.read_excel(
    "/content/sinhala_youtube_comments_for_annotation.xlsx"
)

df = df[['comment', 'label']]

df.dropna(inplace=True)

df['label'] = df['label'].astype(int)

print(df.head())

print()

print(df['label'].value_counts())

                                             comment  label
0                                           පියතුමනි      0
1                                       නියම යි❤❤❤❤❤      0
2                              හිච්චගේ මුණ තමා maru😂      0
3  සානක අයියගෙ වීඩියෝ බලනවා නම් හිනා වෙන්න පුලුවන...      1
4  අයියා වෙඩින් මුද්ද නම් දාගෙනම නේද ටික් ටොක් කර...      0

label
0    1577
1    1471
Name: count, dtype: int64


In [ ]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["comment"].astype(str).tolist(),
    df["label"].tolist(),
    test_size=0.20,
    random_state=42,
    stratify=df["label"]
)

print("Training samples:", len(train_texts))
print("Testing samples :", len(test_texts))

Training samples: 2438
Testing samples : 610


In [ ]:
MODEL_NAME = "state-spaces/mamba-130m-hf"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Tokenizer Loaded!")

config.json:   0%|          | 0.00/895 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.79k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

Tokenizer Loaded!


In [ ]:
train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=64
)

test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=64
)

print("Tokenization Completed!")

Tokenization Completed!


In [ ]:
class SarcasmDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):

        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(self.labels[idx])

        return item

    def __len__(self):

        return len(self.labels)

In [ ]:
train_dataset = SarcasmDataset(
    train_encodings,
    train_labels
)

test_dataset = SarcasmDataset(
    test_encodings,
    test_labels
)

print("Datasets Ready!")

Datasets Ready!


In [ ]:
import transformers
from transformers import MambaModel, MambaConfig

print("Transformers version:", transformers.__version__)
print("MambaModel:", MambaModel)
print("MambaConfig:", MambaConfig)

Transformers version: 5.12.1
MambaModel: <class 'transformers.models.mamba.modeling_mamba.MambaModel'>
MambaConfig: <class 'transformers.models.mamba.configuration_mamba.MambaConfig'>


In [ ]:
from transformers import MambaModel

MODEL_NAME = "state-spaces/mamba-130m-hf"

backbone = MambaModel.from_pretrained(MODEL_NAME)

print("Pretrained Mamba loaded successfully!")

model.safetensors:   0%|          | 0.00/517M [00:00<?, ?B/s]

[transformers] The fast path is not available because one of `(selective_state_update, selective_scan_fn, causal_conv1d_fn, causal_conv1d_update, mamba_inner_fn)` is None. Falling back to the sequential implementation of Mamba, as use_mambapy is set to False. To install follow https://github.com/state-spaces/mamba/#installation for mamba-ssm and install the kernels library using `pip install kernels` or https://github.com/Dao-AILab/causal-conv1d for causal-conv1d. For the mamba.py backend, follow https://github.com/alxndrTL/mamba.py.


Loading weights:   0%|          | 0/242 [00:00<?, ?it/s]

Pretrained Mamba loaded successfully!


In [ ]:
import torch
import torch.nn as nn
from transformers import MambaPreTrainedModel

class MambaClassifier(MambaPreTrainedModel):

    def __init__(self, config):

        super().__init__(config)

        self.mamba = MambaModel(config)

        self.dropout = nn.Dropout(0.1)

        self.classifier = nn.Linear(config.hidden_size, 2)

        self.post_init()

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels=None
    ):

        outputs = self.mamba(
            input_ids=input_ids
        )

        hidden_states = outputs.last_hidden_state

        pooled_output = hidden_states.mean(dim=1)

        pooled_output = self.dropout(pooled_output)

        logits = self.classifier(pooled_output)

        loss = None

        if labels is not None:

            loss_fn = nn.CrossEntropyLoss()

            loss = loss_fn(logits, labels)

        return {
            "loss": loss,
            "logits": logits
        }

In [ ]:
model = MambaClassifier(backbone.config)

print(model)

MambaClassifier(
  (mamba): MambaModel(
    (embeddings): Embedding(50280, 768)
    (layers): ModuleList(
      (0-23): 24 x MambaBlock(
        (norm): MambaRMSNorm(768, eps=1e-05)
        (mixer): MambaMixer(
          (conv1d): Conv1d(1536, 1536, kernel_size=(4,), stride=(1,), padding=(3,), groups=1536)
          (act): SiLUActivation()
          (in_proj): Linear(in_features=768, out_features=3072, bias=False)
          (x_proj): Linear(in_features=1536, out_features=80, bias=False)
          (dt_proj): Linear(in_features=48, out_features=1536, bias=True)
          (out_proj): Linear(in_features=1536, out_features=768, bias=False)
        )
      )
    )
    (norm_f): MambaRMSNorm(768, eps=1e-05)
  )
  (dropout): Dropout(p=0.1, inplace=False)
  (classifier): Linear(in_features=768, out_features=2, bias=True)
)


In [ ]:
model.mamba.load_state_dict(
    backbone.state_dict()
)

print("Backbone copied successfully!")

Backbone copied successfully!


In [ ]:
device = torch.device("cuda")

model.to(device)

print("Using device:", device)

Using device: cuda


In [ ]:
sample = tokenizer(
    "මේක නියම විහිළුවක් 😂",
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=64
)

sample = {
    k: v.to(device)
    for k, v in sample.items()
}

with torch.no_grad():

    outputs = model(**sample)

print(outputs["logits"])

tensor([[ 1.1443, -0.0904]], device='cuda:0')


In [ ]:
from transformers.modeling_outputs import SequenceClassifierOutput

In [ ]:
import torch
import torch.nn as nn

from transformers import MambaModel, MambaPreTrainedModel
from transformers.modeling_outputs import SequenceClassifierOutput


class MambaClassifier(MambaPreTrainedModel):

    def __init__(self, config):

        super().__init__(config)

        # Pretrained Mamba backbone
        self.mamba = MambaModel(config)

        # Dropout
        self.dropout = nn.Dropout(0.1)

        # Classification layer
        self.classifier = nn.Linear(config.hidden_size, 2)

        # Initialize newly added layers
        self.post_init()

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels=None,
    ):

        outputs = self.mamba(
            input_ids=input_ids
        )

        # Last hidden state
        hidden_states = outputs.last_hidden_state

        # Mean Pooling
        pooled_output = hidden_states.mean(dim=1)

        # Dropout
        pooled_output = self.dropout(pooled_output)

        # Classification logits
        logits = self.classifier(pooled_output)

        loss = None

        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits
        )

In [ ]:
model = MambaClassifier(backbone.config)

model.mamba.load_state_dict(
    backbone.state_dict()
)

device = torch.device("cuda")

model.to(device)

print("Model rebuilt successfully!")

Model rebuilt successfully!


In [ ]:
model.gradient_checkpointing_enable()

In [ ]:
sample = tokenizer(
    "මේක නියම විහිළුවක් 😂",
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=64
)

sample = {k: v.to(device) for k, v in sample.items()}

with torch.no_grad():
    outputs = model(**sample)

print(outputs.logits)

tensor([[-1.9388, -3.3462]], device='cuda:0')


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = logits.argmax(axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro"
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

print("Metrics function ready!")

Metrics function ready!


In [ ]:
training_args = TrainingArguments(

    output_dir="./mamba_results",

    learning_rate=2e-5,

    num_train_epochs=3,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    gradient_accumulation_steps=2,

    fp16=True,

    weight_decay=0.01,

    logging_steps=50,

    save_strategy="epoch",

    eval_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    report_to="none",

    save_total_limit=1
)

In [ ]:
from transformers import EarlyStoppingCallback

early_stopping = EarlyStoppingCallback(
    early_stopping_patience=2
)

print("Early stopping ready!")

Early stopping ready!


In [ ]:
from transformers import Trainer

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    compute_metrics=compute_metrics,

    callbacks=[early_stopping]
)

print("Trainer created successfully!")

Trainer created successfully!


In [ ]:
import gc
import torch

gc.collect()

torch.cuda.empty_cache()

torch.cuda.reset_peak_memory_stats()

print(torch.cuda.memory_allocated()/1024**3)

0.4906644821166992


In [ ]:
train_output = trainer.train()

print(train_output)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.700264,0.695553,0.537705,0.570843,0.522895,0.433130
2,0.638432,0.664244,0.586885,0.589469,0.581611,0.575002
3,0.602755,0.655616,0.619672,0.619615,0.617164,0.616440


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=459, training_loss=0.6680794312803314, metrics={'train_runtime': 3320.6111, 'train_samples_per_second': 2.203, 'train_steps_per_second': 0.138, 'total_flos': 254237517854208.0, 'train_loss': 0.6680794312803314, 'epoch': 3.0})


In [ ]:
results = trainer.evaluate()

print(results)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.602755,0.655616,3,0.619672,0.619615,0.617164,0.616440


{'eval_loss': 0.6556164622306824, 'eval_accuracy': 0.6196721311475409, 'eval_precision': 0.6196153846153847, 'eval_recall': 0.6171639541892706, 'eval_f1': 0.6164395442373782}


In [ ]:
predictions = trainer.predict(test_dataset)

preds = np.argmax(predictions.predictions, axis=1)

labels = predictions.label_ids

In [ ]:
cm = confusion_matrix(labels, preds)

print(cm)

[[217  99]
 [133 161]]


In [ ]:
print(classification_report(labels, preds))

              precision    recall  f1-score   support

           0       0.62      0.69      0.65       316
           1       0.62      0.55      0.58       294

    accuracy                           0.62       610
   macro avg       0.62      0.62      0.62       610
weighted avg       0.62      0.62      0.62       610



In [ ]:
accuracy = accuracy_score(labels, preds)

print("Accuracy :", accuracy)

Accuracy : 0.6196721311475409


In [ ]:
precision, recall, f1, _ = precision_recall_fscore_support(
    labels,
    preds,
    average="macro"
)

print("Precision :", precision)
print("Recall :", recall)
print("Macro F1 :", f1)

Precision : 0.6196153846153847
Recall : 0.6171639541892706
Macro F1 : 0.6164395442373782


In [ ]:
precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(
    labels,
    preds,
    average="weighted"
)

print("Weighted F1 :", f1_w)

Weighted F1 : 0.6177094890949422


In [ ]:
trainer.save_model("./Mamba_Sinhala_Model")

tokenizer.save_pretrained("./Mamba_Sinhala_Model")

print("Model Saved Successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model Saved Successfully!


In [ ]:
def predict(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        outputs = model(**inputs)

        prediction = torch.argmax(outputs.logits, dim=1).item()

    return prediction

In [ ]:
sentence = "අනේ ඔයා නම් හරිම දක්ෂයි."

prediction = predict(sentence)

print(prediction)

0


In [ ]:
examples = [

    "අනේ ඔයා නම් හරිම දක්ෂයි.",

    "අද ගොඩක් ලස්සන දවසක්.",

    "මේක තමයි හොඳම වැඩේ.",

    "ඔයා කරපු වැඩේ නම් නියමයි."

]

for text in examples:

    print(text)

    print("Prediction :", predict(text))

    print("-"*50)

අනේ ඔයා නම් හරිම දක්ෂයි.
Prediction : 0
--------------------------------------------------
අද ගොඩක් ලස්සන දවසක්.
Prediction : 1
--------------------------------------------------
මේක තමයි හොඳම වැඩේ.
Prediction : 0
--------------------------------------------------
ඔයා කරපු වැඩේ නම් නියමයි.
Prediction : 0
--------------------------------------------------
